# Lab 2: Naive Bayes Classifier — Traffic Data (Multi-label Classification)

## 1. Objective
Build a **Naive Bayes classifier** to predict the traffic status (`Class`) — one of `On Time`, `Late`, `Very Late`, or `Cancelled` — based on the attributes **Day**, **Season**, **Fog**, and **Rain**.

## 2. Training Dataset

| Day      | Season | Fog    | Rain  | Class     |
|----------|--------|--------|-------|-----------|
| Weekday  | Spring | None   | None  | On Time   |
| Weekday  | Winter | None   | Slight| On Time   |
| Weekday  | Winter | None   | None  | On Time   |
| Holiday  | Winter | High   | Slight| Late      |
| Saturday | Summer | Normal | None  | On Time   |
| Weekday  | Autumn | Normal | None  | Very Late |
| Holiday  | Summer | High   | Slight| On Time   |
| Sunday   | Summer | Normal | None  | On Time   |
| Weekday  | Winter | High   | Heavy | Very Late |
| Weekday  | Summer | None   | Slight| On Time   |
| Saturday | Spring | High   | Heavy | Cancelled |
| Weekday  | Summer | High   | Slight| On Time   |
| Weekday  | Winter | Normal | None  | Late      |
| Weekday  | Summer | High   | None  | On Time   |
| Weekday  | Winter | Normal | Heavy | Vary Late |
| Saturday | Autumn | High   | Slight| On Time   |
| Weekday  | Autumn | None   | Heavy | On Time   |
| Holiday  | Spring | Normal | Slight| On Time   |
| Weekday  | Spring | Normal | None  | On Time   |
| Weekday  | Spring | Normal | Heavy | On Time   |

## 3. Test Event

$$
X = (\text{Day} = \text{Weekday}, \ \text{Season} = \text{Winter}, \ \text{Fog} = \text{High}, \ \text{Rain} = \text{Heavy})
$$

**Question:** Compute the probability of each class — `On Time`, `Late`, `Very Late`, and `Cancelled` — given the event $X$ above.

## 4. Theoretical Background

By **Bayes' Theorem**:

$$
P(c \mid X) = \frac{P(X \mid c) \cdot P(c)}{P(X)}
$$

where:
- $c$ — class label ($c \in \{\text{On Time}, \text{Late}, \text{Very Late}, \text{Cancelled}\}$)
- $X = \{x_1, x_2, x_3, x_4\}$ — feature vector (Day, Season, Fog, Rain)

Since $P(X)$ is constant for all classes, comparing classes only requires the numerator:

$$
P(c \mid X) \propto P(X \mid c) \cdot P(c)
$$

Applying the **Naive Bayes assumption** — that features are conditionally independent given the class — we get:

$$
P(c \mid X) \propto P(c) \cdot \prod_{i=1}^{n} P(x_i \mid c)
$$

$$
P(c \mid X) \propto P(c) \cdot P(\text{Day} \mid c) \cdot P(\text{Season} \mid c) \cdot P(\text{Fog} \mid c) \cdot P(\text{Rain} \mid c)
$$

## 5. Classification Steps
1. Compute the prior probability $P(c)$ for each class (`On Time`, `Late`, `Very Late`, `Cancelled`) from the training set.
2. Compute the conditional probability of each feature value given each class, e.g. $P(\text{Day} = \text{Weekday} \mid \text{On Time})$.
3. For the test event $X$, compute the (unnormalized) posterior score for each of the 4 classes using the product formula above.
4. Compare the scores to determine which class $X$ is most likely to belong to.

In [186]:
pip install numpy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [187]:
import math
import numpy as np

In [188]:
def create_train_data():
    train_data = [
        ["Weekday", "Spring", "None", "None", "On Time"],
        ["Weekday", "Winter", "None", "Slight", "On Time"],
        ["Weekday", "Winter", "None", "None", "On Time"],
        ["Holiday", "Winter", "High", "Slight", "Late"],
        ["Saturday", "Summer", "Normal", "None", "On Time"],
        ["Weekday", "Autumn", "Normal", "None", "Very Late"],
        ["Holiday", "Summer", "High", "Slight", "On Time"],
        ["Sunday", "Summer", "Normal", "None", "On Time"],
        ["Weekday", "Winter", "High", "Heavy", "Very Late"],
        ["Weekday", "Summer", "None", "Slight", "On Time"],
        ["Saturday", "Spring", "High", "Heavy", "Cancelled"],
        ["Weekday", "Summer", "High", "Slight", "On Time"],
        ["Weekday", "Winter", "Normal", "None", "Late"],
        ["Weekday", "Summer", "High", "None", "On Time"],
        ["Weekday", "Winter", "Normal", "Heavy", "Very Late"],
        ["Saturday", "Autumn", "High", "Slight", "On Time"],
        ["Weekday", "Autumn", "None", "Heavy", "On Time"],
        ["Holiday", "Spring", "Normal", "Slight", "On Time"],
        ["Weekday", "Spring", "Normal", "None", "On Time"],
        ["Weekday", "Spring", "Normal", "Heavy", "On Time"]
    ]
    return np.array(train_data)


train_data = create_train_data()
print(train_data)

[['Weekday' 'Spring' 'None' 'None' 'On Time']
 ['Weekday' 'Winter' 'None' 'Slight' 'On Time']
 ['Weekday' 'Winter' 'None' 'None' 'On Time']
 ['Holiday' 'Winter' 'High' 'Slight' 'Late']
 ['Saturday' 'Summer' 'Normal' 'None' 'On Time']
 ['Weekday' 'Autumn' 'Normal' 'None' 'Very Late']
 ['Holiday' 'Summer' 'High' 'Slight' 'On Time']
 ['Sunday' 'Summer' 'Normal' 'None' 'On Time']
 ['Weekday' 'Winter' 'High' 'Heavy' 'Very Late']
 ['Weekday' 'Summer' 'None' 'Slight' 'On Time']
 ['Saturday' 'Spring' 'High' 'Heavy' 'Cancelled']
 ['Weekday' 'Summer' 'High' 'Slight' 'On Time']
 ['Weekday' 'Winter' 'Normal' 'None' 'Late']
 ['Weekday' 'Summer' 'High' 'None' 'On Time']
 ['Weekday' 'Winter' 'Normal' 'Heavy' 'Very Late']
 ['Saturday' 'Autumn' 'High' 'Slight' 'On Time']
 ['Weekday' 'Autumn' 'None' 'Heavy' 'On Time']
 ['Holiday' 'Spring' 'Normal' 'Slight' 'On Time']
 ['Weekday' 'Spring' 'Normal' 'None' 'On Time']
 ['Weekday' 'Spring' 'Normal' 'Heavy' 'On Time']]


In [189]:
def compute_prior_probs(train_data):
    unique_class = np.unique(train_data[:, 4])
    sample_class = len(train_data)
    prior_probs = np.zeros(len(unique_class))
    
    for class_idx, class_name in enumerate(unique_class):
        for train_val in train_data:
            if (train_val[4] == class_name):
                prior_probs[class_idx] += 1
        prior_probs[class_idx] /= sample_class
    
    return unique_class, prior_probs

class_names = compute_prior_probs(train_data)[0]
prior_probs = compute_prior_probs(train_data)[1]
print(class_names)
print()
print(compute_prior_probs(train_data)[1])

['Cancelled' 'Late' 'On Time' 'Very Late']

[0.05 0.1  0.7  0.15]


In [190]:
def compute_conditional_probs(train_data):
    n_feat = train_data.shape[1] - 1
    cond_probs = []
    feature_values = []
    
    for i in range(n_feat): # X1 X2 X3 X4
        #Xi
        unique_val = np.unique(train_data[:, i])
        feature_values.append(unique_val)
        feat_cond_probs = np.zeros((len(class_names), len(unique_val)))
        #print(feat_cond_probs.shape)
        #print(len(class_names))
        #print(unique_val)
        
        all_feat = np.array(train_data[:, [i, 4]])
        #print(all_val)
        
        for prior_idx, prior_val in enumerate(prior_probs):
            for u_idx, u_val in enumerate(unique_val):
                cnt = 0
                for val in all_feat:
                    if (val[0] == u_val and val[1] == class_names[prior_idx]):
                        cnt += 1
                feat_cond_probs[prior_idx][u_idx] = cnt / (prior_val * 20)
                
        cond_probs.append(feat_cond_probs)
                
    
    return cond_probs, feature_values

conditional_probs, feat_val = compute_conditional_probs(train_data)

print(feat_val)
print(conditional_probs)

[array(['Holiday', 'Saturday', 'Sunday', 'Weekday'], dtype='<U9'), array(['Autumn', 'Spring', 'Summer', 'Winter'], dtype='<U9'), array(['High', 'None', 'Normal'], dtype='<U9'), array(['Heavy', 'None', 'Slight'], dtype='<U9')]
[array([[0.        , 1.        , 0.        , 0.        ],
       [0.5       , 0.        , 0.        , 0.5       ],
       [0.14285714, 0.14285714, 0.07142857, 0.64285714],
       [0.        , 0.        , 0.        , 1.        ]]), array([[0.        , 1.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 1.        ],
       [0.14285714, 0.28571429, 0.42857143, 0.14285714],
       [0.33333333, 0.        , 0.        , 0.66666667]]), array([[1.        , 0.        , 0.        ],
       [0.5       , 0.        , 0.5       ],
       [0.28571429, 0.35714286, 0.35714286],
       [0.33333333, 0.        , 0.66666667]]), array([[1.        , 0.        , 0.        ],
       [0.        , 0.5       , 0.5       ],
       [0.14285714, 0.42857143, 0.4285714

In [191]:
def get_feature_index(feature_value, feature_values):
  for idx, val in enumerate(feature_values):
    if (val == feature_value):
      return idx


_, feature_values = compute_conditional_probs(train_data)
outlook = feature_values[0]

i1 = get_feature_index("Weekday", outlook)
i2 = get_feature_index("Holiday", outlook)
i3 = get_feature_index("Sunday", outlook)

print(i1, i2, i3)

3 0 2


In [192]:
def train_modal(train_data):
    prior_probabilities = compute_prior_probs(train_data)
    
    conditional_probabilities, feature_names = compute_conditional_probs(train_data)
    
    return prior_probabilities, conditional_probabilities, feature_names
    
prior_probs, conditional_probs, feature_names =  train_modal(
train_data)

print(prior_probs)
print(conditional_probs)
print(feature_names)

(array(['Cancelled', 'Late', 'On Time', 'Very Late'], dtype='<U9'), array([0.05, 0.1 , 0.7 , 0.15]))
[array([[0.        , 1.        , 0.        , 0.        ],
       [0.5       , 0.        , 0.        , 0.5       ],
       [0.14285714, 0.14285714, 0.07142857, 0.64285714],
       [0.        , 0.        , 0.        , 1.        ]]), array([[0.        , 1.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 1.        ],
       [0.14285714, 0.28571429, 0.42857143, 0.14285714],
       [0.33333333, 0.        , 0.        , 0.66666667]]), array([[1.        , 0.        , 0.        ],
       [0.5       , 0.        , 0.5       ],
       [0.28571429, 0.35714286, 0.35714286],
       [0.33333333, 0.        , 0.66666667]]), array([[1.        , 0.        , 0.        ],
       [0.        , 0.5       , 0.5       ],
       [0.14285714, 0.42857143, 0.42857143],
       [0.66666667, 0.33333333, 0.        ]])]
[array(['Holiday', 'Saturday', 'Sunday', 'Weekday'], dtype='<U9'), array([

In [193]:
def predict(X, prior_probabilities, conditional_probabilities, feature_names):
    feature_indices = []
    for i, feature_value in enumerate(X):
        feature_indices.append(get_feature_index(feature_value, feature_names[i]))

    # print(len(prior_probabilities[1]))

    #print(conditional_probabilities[Xi][class_name][Feature])

    #print(feature_indices)
    
    # print(conditional_probabilities[0][2][0])
    class_probs = np.zeros(len(class_names))
    for i in range(len(class_names)):
        class_probs[i] = (prior_probabilities[1][i])
        for feat_id, feat_val in enumerate(feature_indices):
            class_probs[i] *= conditional_probabilities[feat_id][i][feat_val]
    
    #print(class_probs)
    
    #normalize
    total_probs = sum(class_probs)
    if (total_probs > 0):
        normalize_probs = [p / total_probs for p in class_probs]
    else:
        normalize_probs = [0.25, 0.25, 0.25, 0.25]
    
    for i in range(len(normalize_probs)):
        normalize_probs[i] = round(normalize_probs[i].item(), 2),
    # predict
    predicted_class_idx = np.argmax(class_probs)
    prediction = class_names[predicted_class_idx]
    return prediction, normalize_probs
predict(["Weekday", "Winter", "High", "Heavy"], prior_probs, conditional_probs, feature_names)

(np.str_('Very Late'), [(0.0,), (0.0,), (0.11,), (0.89,)])

In [194]:
def result(X):    
    print("Data test: ", end="")
    print(X)
    predict_val, probs = predict(X, prior_probs, conditional_probs, feature_names)
    print("Probabilities: ")
    for i in range(len(class_names)):
        print(class_names[i], end=": ")
        print(float(probs[i][0]) * 100, "%")
    print("Prediction: ", predict_val)

X = ["Weekday", "Winter", "High", "Heavy"]
result(X)

Data test: ['Weekday', 'Winter', 'High', 'Heavy']
Probabilities: 
Cancelled: 0.0 %
Late: 0.0 %
On Time: 11.0 %
Very Late: 89.0 %
Prediction:  Very Late
